# 04 · Extensión (PoC): comentario del hallazgo con **MedGemma**

Prueba de concepto: combinar la imagen + la probabilidad de nuestro clasificador
y pedir a **MedGemma** (modelo médico multimodal abierto de Google) un comentario
textual del hallazgo.

> 🚫 **No se despliega en el Space gratuito** (MedGemma es grande y necesita GPU).
> Esto corre en **Colab con GPU**.
>
> 🔐 Requiere **aceptar los términos** del modelo en Hugging Face
> (`google/medgemma-4b-it`) y un token de HF.
>
> ⚠️ **El texto generado es ILUSTRATIVO, no un diagnóstico.** No usar con pacientes reales.

## 1 · Instalar y autenticarse

In [1]:
# Setup automático en Google Colab (se omite si se corre localmente)
import os, sys

if "google.colab" in sys.modules:
    repo_dir = "/content/fracturas-rayos-x"
    if not os.path.isdir(repo_dir):
        os.system("git clone https://github.com/stevenrq/fracturas-rayos-x.git " + repo_dir)
    os.chdir(repo_dir)
    print("Directorio de trabajo:", os.getcwd())

Directorio de trabajo: /content/fracturas-rayos-x


In [2]:
# !pip -q install -U transformers accelerate huggingface_hub
from huggingface_hub import login
login()   # pega tu token de HF (con acceso aceptado a google/medgemma-4b-it)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


## 2 · Cargar MedGemma (multimodal)

In [4]:
import torch
from transformers import pipeline

pipe = pipeline(
    "image-text-to-text",
    model="google/medgemma-4b-it",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [5]:
import os
import subprocess
from pathlib import Path
import numpy as np
from PIL import Image
from tensorflow import keras

# Encontrar la raíz del repo de forma robusta (local y Colab)
def _find_root():
    try:
        out = subprocess.check_output(
            ["git", "rev-parse", "--show-toplevel"], stderr=subprocess.DEVNULL
        )
        return Path(out.decode().strip())
    except Exception:
        pass
    # Buscar hacia arriba desde cwd hasta encontrar la carpeta app/
    p = Path(os.getcwd())
    for _ in range(5):
        if (p / "app" / "ejemplos").exists():
            return p
        p = p.parent
    return Path(os.getcwd())

_root = _find_root()
print("Raíz del repo:", _root)

IMG = str(_root / "app/ejemplos/fractura.jpg")   # cambia por tu radiografía
pil = Image.open(IMG).convert("RGB")

p_fractura = None
try:
    m = keras.models.load_model(str(_root / "models/best_mobilenet.keras"))
    arr = np.asarray(pil.resize((224, 224)), dtype="float32")[None, ...]
    p_fractura = float(m.predict(arr, verbose=0).ravel()[0])
    print(f"P(fractura) del modelo = {p_fractura:.1%}")
except Exception as e:
    print("Sin modelo binario disponible (se omite la pista):", e)

Raíz del repo: /content/fracturas-rayos-x
Sin modelo binario disponible (se omite la pista): File not found: filepath=/content/fracturas-rayos-x/models/best_mobilenet.keras. Please ensure the file is an accessible `.keras` zip file.


## 4 · Pedir el comentario a MedGemma

In [6]:
pista = (f" Un clasificador CNN estima P(fractura)={p_fractura:.0%}."
         if p_fractura is not None else "")
prompt = ("Describe brevemente los hallazgos de esta radiografía y si hay signos "
          "de fractura." + pista + " Responde en español, en 3-4 frases.")

messages = [
    {"role": "system", "content": [{"type": "text",
        "text": "Eres un asistente experto en radiología. Tu salida es educativa, no un diagnóstico."}]},
    {"role": "user", "content": [
        {"type": "text", "text": prompt},
        {"type": "image", "image": pil},
    ]},
]

out = pipe(text=messages, max_new_tokens=300)
print(out[0]["generated_text"][-1]["content"])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


La radiografía muestra el fémur y la tibia, con la rodilla visible en el centro. No se observan signos evidentes de fractura en el fémur o la tibia. La densidad ósea parece normal en la región examinada.



## 5 · Aviso

El comentario anterior es una **demostración** de un modelo de lenguaje. **No** es
un diagnóstico, puede contener errores ("alucinaciones") y **no** debe usarse en
decisiones clínicas reales. Por eso esta capacidad se mantiene como PoC y **no**
forma parte de la app desplegada.